# TripAdvisor Recommender System
**Authors:** Houssam ElMouedden & [Partner's Name]
**Date:** March 2026

## Project Overview
The goal of this project is to recommend similar places using only TripAdvisor reviews. 
* **The Idea:** If two places are described with the same words, they are probably similar.
* **The Rule:** We can only use the review text to make recommendations. We use the extra details (metadata like price or category) just to grade how well our model works.

In [27]:
# Imports and Setup
import pandas as pd
import numpy as np
import os
import string
import ast
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi

# Set random seed for reproducibility
np.random.seed(42)

## 1. Data Loading
First, we load our two datasets. To make things simpler, we filter the data to keep only the English reviews. Then, we connect the reviews to the correct places using their IDs.

In [ ]:
# Define paths
data_dir = r"C:\Users\testd\Desktop\Esilv-Data engineering & AI\Information Retrieval and NLP\TripAdvisor-NLP-Recommender\TripAdvisorTrainingDataProject1"
reviews_path = os.path.join(data_dir, "reviews83325.csv")
places_path = os.path.join(data_dir, "Tripadvisor.csv")

Loading data...


C:\Users\testd\AppData\Local\Temp\ipykernel_27640\1133754036.py:8: DtypeWarning: Columns (15,16,17,18,19,20) have mixed types. Specify dtype option on import or set low_memory=False.
  df_reviews = pd.read_csv(reviews_path)


Total English reviews: 153071


In [ ]:
# Load datasets
print("Loading data...")
df_reviews = pd.read_csv(reviews_path)
df_places = pd.read_csv(places_path)

In [ ]:
# Filter for English reviews only
df_reviews = df_reviews[df_reviews['langue'] == 'en'].copy()
print(f"Total English reviews: {len(df_reviews)}")

## 2. Preparing the Data
We split the unique places evenly into training (queries) and testing (search corpus). To handle the varying number of reviews per place, we aggregate all reviews for a single place into one large text block.

In [30]:
print(df_reviews.columns)

Index(['id', 'idplace', 'titre', 'idauteur', 'review', 'note', 'date_review',
       'date_visit', 'langue', 'published_platform', 'typeReview',
       'subratings', 'machine_translated', 'machine_translatable', 'owner_id',
       'owner_langue', 'owner_date_review', 'owner_connection',
       'owner_responder', 'owner_response', 'owner_title'],
      dtype='object')


In [37]:
# Random split of places (50% train, 50% test)
train_places, test_places = train_test_split(df_places, test_size=0.5, random_state=42)

# Group reviews by place into a single document using the correct 'review' column
review_col = 'review' 
grouped_reviews = df_reviews.groupby('idplace')[review_col].apply(lambda x: ' '.join(x.dropna().astype(str))).reset_index()

# Merge the grouped text with the metadata
train_data = pd.merge(train_places, grouped_reviews, left_on='id', right_on='idplace', how='inner')
test_data = pd.merge(test_places, grouped_reviews, left_on='id', right_on='idplace', how='inner')

print(f"Train queries available: {len(train_data)}")
print(f"Test corpus size: {len(test_data)}")

Train queries available: 908
Test corpus size: 927


## 3. Evaluation Metrics Setup
We calculate a "Ranking Error" based on the list position of the first correctly matched recommendation. The lower the score, the better the recommendation. We check for broad category matches (Level 1) and specific sub-category matches (Level 2).

In [ ]:
def check_intersection(val1, val2):
    """Checks if two list-like strings have common elements."""
    if pd.isna(val1) or pd.isna(val2): return False
    try:
        list1 = ast.literal_eval(val1) if isinstance(val1, str) and '[' in val1 else [val1]
        list2 = ast.literal_eval(val2) if isinstance(val2, str) and '[' in val2 else [val2]
        return len(set(list1).intersection(set(list2))) > 0
    except:
        return val1 == val2

def is_match_level_1(query_row, retrieved_row):
    return query_row.get('typeR') == retrieved_row.get('typeR')

def is_match_level_2(query_row, retrieved_row):
    if not is_match_level_1(query_row, retrieved_row): return False
    
    place_type = query_row.get('typeR')
    if place_type in ['A', 'AP']:
        return (check_intersection(query_row.get('activiteSubCategorie'), retrieved_row.get('activiteSubCategorie')) or 
                check_intersection(query_row.get('activiteSubType'), retrieved_row.get('activiteSubType')))
    elif place_type == 'R':
        return (check_intersection(query_row.get('restaurantType'), retrieved_row.get('restaurantType')) or 
                check_intersection(query_row.get('restaurantTypeCuisine'), retrieved_row.get('restaurantTypeCuisine')))
    elif place_type == 'H':
        return query_row.get('priceRange') == retrieved_row.get('priceRange')
    return False

def calculate_ranking_error(query_row, ranked_test_df, level=1):
    """Calculates ranking error based on the position of the first match."""
    match_func = is_match_level_1 if level == 1 else is_match_level_2
    
    # If no match exists in the test set, error is undefined
    possible_matches = [i for i, row in ranked_test_df.iterrows() if match_func(query_row, row)]
    if not possible_matches: return np.nan 

    for idx, row in ranked_test_df.iterrows():
        if match_func(query_row, row): return idx 
    return np.nan

## 4. The Baseline Model (BM25)
Before building our own model, we first test a standard search tool called BM25. This gives us a baseline error score that we need to try and beat.

In [ ]:
def clean_and_tokenize(text):
    if not isinstance(text, str): return []
    text = text.lower().translate(str.maketrans('', '', string.punctuation))
    return text.split()

print("Tokenizing corpus for BM25...")
test_tokens = test_data[review_col].apply(clean_and_tokenize).tolist()

bm25 = BM25Okapi(test_tokens)

def evaluate_model_bm25(train_df, test_df, num_queries=50):
    errors_l1, errors_l2 = [], []
    queries = train_df.head(num_queries)
    
    for _, query_row in queries.iterrows():
        query_tokens = clean_and_tokenize(query_row[review_col])
        scores = bm25.get_scores(query_tokens)
        
        ranked_indices = np.argsort(scores)[::-1]
        ranked_test_df = test_df.iloc[ranked_indices].reset_index(drop=True)
        
        err_l1 = calculate_ranking_error(query_row, ranked_test_df, level=1)
        err_l2 = calculate_ranking_error(query_row, ranked_test_df, level=2)
        
        if not np.isnan(err_l1): errors_l1.append(err_l1)
        if not np.isnan(err_l2): errors_l2.append(err_l2)
            
    return np.mean(errors_l1), np.mean(errors_l2)

print("Evaluating BM25...")
bm25_l1, bm25_l2 = evaluate_model_bm25(train_data, test_data)
print(f"BM25 Mean Ranking Error (Level 1): {bm25_l1:.2f}")
print(f"BM25 Mean Ranking Error (Level 2): {bm25_l2:.2f}")

Tokenizing corpus for BM25...
Evaluating BM25...
BM25 Mean Ranking Error (Level 1): 0.44
BM25 Mean Ranking Error (Level 2): 40.00


## 5. Deliverable 2: Custom TF-IDF Model
We build a custom model using TF-IDF and Cosine Similarity. We limit the vocabulary to the top 5000 features to reduce noise and focus on the most impactful descriptive words.

In [45]:
print("Fitting TF-IDF Model...")
tfidf = TfidfVectorizer(stop_words='english', max_features=5000, lowercase=True)

# Fit on the test corpus
test_tfidf_matrix = tfidf.fit_transform(test_data[review_col].fillna(''))

def evaluate_custom_model(train_df, test_df, num_queries=50):
    errors_l1, errors_l2 = [], []
    queries = train_df.head(num_queries)
    
    # Transform queries into the TF-IDF space
    query_tfidf_matrix = tfidf.transform(queries[review_col].fillna(''))
    
    sim_scores = cosine_similarity(query_tfidf_matrix, test_tfidf_matrix)
    
    for i, (_, query_row) in enumerate(queries.iterrows()):
        scores = sim_scores[i]
        ranked_indices = np.argsort(scores)[::-1]
        ranked_test_df = test_df.iloc[ranked_indices].reset_index(drop=True)
        
        err_l1 = calculate_ranking_error(query_row, ranked_test_df, level=1)
        err_l2 = calculate_ranking_error(query_row, ranked_test_df, level=2)
        
        if not np.isnan(err_l1): errors_l1.append(err_l1)
        if not np.isnan(err_l2): errors_l2.append(err_l2)
            
    return np.mean(errors_l1), np.mean(errors_l2)

print("Evaluating Custom Model...")
custom_l1, custom_l2 = evaluate_custom_model(train_data, test_data, num_queries=50)
print(f"Custom Model Mean Ranking Error (Level 1): {custom_l1:.2f}")
print(f"Custom Model Mean Ranking Error (Level 2): {custom_l2:.2f}")

Fitting TF-IDF Model...
Evaluating Custom Model...
Custom Model Mean Ranking Error (Level 1): 0.68
Custom Model Mean Ranking Error (Level 2): 8.27


In [ ]:
print(df_places.columns)

Index(['id', 'idTrip', 'fromId', 'nom', 'url', 'rating', 'nbAvis',
       'nbAvisRecupere', 'latitude', 'longitude', 'typeR', 'adresse',
       'priceRange', 'closed', 'hotelType', 'hotelStyle', 'hotelStars',
       'hotelRoomNumber', 'hotelNoteEmplacement', 'hotelNoteProprete',
       'hotelNoteService', 'HotelNoteQualitePrix', 'hoteldistance',
       'hotelbearing', 'restaurantTypeCuisine',
       'restaurantDietaryRestrictions', 'restaurantMeals',
       'restaurantFeatures', 'restaurantNoteCuisine', 'restaurantNoteService',
       'restaurantNoteQualitePrix', 'restaurantNoteAmbiance', 'activiteType',
       'activiteSubCategorie', 'activiteSubType', 'website', 'nbScanReview',
       'dateLastScanReviews', 'shape_gid', 'gadm36_gid', 'hotelprice',
       'hotelBookingID', 'restaurantSubcategory', 'restaurantType',
       'ap_additional_info', 'ap_age_band_list', 'ap_attraction_ids',
       'ap_booking_question_list', 'ap_bubble_rating_integer', 'ap_duration',
       'ap_exclusion', '

: 